In [1]:
import pandas as pd
import numpy as np
import os
from joblib import Parallel, delayed
import statsmodels.api as sm
from tqdm import tqdm

In [2]:
data_path = '/data/GRACE_data/A_share'
FF5_df = pd.read_csv(f'{data_path}/FF_5.csv')
ymd = pd.to_datetime(FF5_df['date'], format='%Y/%m/%d')
FF5_df['date'] = ymd.apply(lambda x: f'{x.year}-{x.month:02d}-{x.day:02d}')
FF5_df.set_index('date', inplace = True)
FF5_df.sort_index(inplace = True)

In [3]:
FF5_df['Mkt-RF'] = FF5_df['MKT'] - FF5_df['RF']

In [4]:
valid_stock_list = np.unique(pd.read_csv(f'{data_path}/stock_pool.csv', index_col = 'date').values.reshape(-1)).tolist()

In [5]:
def calculate_factor_loading(regression_df):
    regression_df = regression_df[['ret', 'interception', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']].copy()
    regression_df.dropna(inplace = True)
    ols_model = sm.OLS(regression_df['ret'], regression_df[['interception'] + ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']])
    ols_result = ols_model.fit()
    return ols_result.params[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']].values

In [6]:
def sub_job(stock, FF5_df, save_path):
    kline_df = pd.read_csv(f'{data_path}/kline_day/{stock}.csv', index_col = 'date')
    kline_df['ret'] = kline_df['close']/kline_df['close'].shift(1) - 1
    kline_df[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']] = FF5_df[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]
    kline_df['interception'] = 1.0
    factor_loading_list = [f'{x}_loading' for x in ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]
    kline_df[[factor_loading_list]] = np.nan
    for i, date in enumerate(kline_df.index.values):
        # kline_df.loc[date, factor_loading_list] = calculate_factor_loading(kline_df.iloc[i - 120:i, :])
        try:
            kline_df.loc[date, factor_loading_list] = calculate_factor_loading(kline_df.iloc[i - 120:i, :])
        except Exception as e:
            continue
    kline_df.to_csv(f'{save_path}/{stock}.csv', columns=factor_loading_list)

In [7]:
result = Parallel(n_jobs=30)(delayed(sub_job)
                                 (stock, FF5_df, f'{data_path}/FF5_factor_loading')
                                 for stock in tqdm(valid_stock_list))

100%|██████████| 1682/1682 [17:17<00:00,  1.62it/s]
